In [ ]:
# ============================================================
# 01. Import libraries
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import Dataset, DatasetDict

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

In [ ]:
# ============================================================
# 02. Set random seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

In [ ]:
# ============================================================
# 03. Experiment configuration
# ============================================================

MODEL_NAME = "sentence-transformers/LaBSE"

TEXT_COLUMN = "noisy_text"
LABEL_COLUMN = "label"

NUM_LABELS = 2

MAX_LENGTH = 128

# Checkpoint saving directory
OUTPUT_DIR = ""

# Trained model saving directory
FINAL_MODEL_DIR = ""

# Results saving directory
RESULTS_PATH = ""

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_LENGTH)
print("Number of labels:", NUM_LABELS)

In [ ]:
# ============================================================
# 04. Dataset paths
# ============================================================

# Train and test datasets of the domain the model is fine-tuned on
TRAIN_PATH = ""
TEST_PATH = ""

# Test datasets of the domain the model is evaluated on
TEST_CLEAN_SINHALA = ""
TEST_TRANSLITERATION = ""
TEST_CODE_MIX = ""
TEST_MISSPELL = ""
TEST_COMPOUND = ""

In [ ]:
# ============================================================
# 05. Load JSONL datasets
# ============================================================

train_df = pd.read_json(
    TRAIN_PATH,
    lines=True
)

test_df = pd.read_json(
    TEST_PATH,
    lines=True
)

test_transliteration_df = pd.read_json(
    TEST_TRANSLITERATION,
    lines=True
)

test_codemix_df = pd.read_json(
    TEST_CODE_MIX,
    lines=True
)

test_clean_sinhala_df = pd.read_json(
    TEST_CLEAN_SINHALA,
    lines=True
)

test_misspell_df = pd.read_json(
    TEST_MISSPELL,
    lines=True
)

test_compound_df = pd.read_json(
    TEST_COMPOUND,
    lines=True
)

print("Original train:", train_df.shape)
print("Original test:", test_df.shape)

print("Transliteration test:", test_transliteration_df.shape)
print("Code-mix test:", test_codemix_df.shape)
print("Clean Sinhala test:", test_clean_sinhala_df.shape)
print("Misspell test:", test_misspell_df.shape)
print("Compound test:", test_compound_df.shape)

In [ ]:
# ============================================================
# 06. Handle text column names
# ============================================================

if "text" in train_df.columns:
    train_df = train_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_df.columns:
    test_df = test_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_transliteration_df.columns:
    test_transliteration_df = test_transliteration_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_codemix_df.columns:
    test_codemix_df = test_codemix_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_clean_sinhala_df.columns:
    test_clean_sinhala_df = test_clean_sinhala_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_misspell_df.columns:
    test_misspell_df = test_misspell_df.rename(
        columns={"text": TEXT_COLUMN}
    )

if "text" in test_compound_df.columns:
    test_compound_df = test_compound_df.rename(
        columns={"text": TEXT_COLUMN}
    )

In [ ]:
# ============================================================
# 07. Split training data into train + validation
# ============================================================

train_df, valid_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nValidation label distribution:")
print(valid_df[LABEL_COLUMN].value_counts())

print("\nTest label distribution:")
print(test_df[LABEL_COLUMN].value_counts())

In [ ]:
# ============================================================
# 08. Keep required columns and clean types
# ============================================================

train_df = train_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

valid_df = valid_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_df = test_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_transliteration_df = test_transliteration_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_codemix_df = test_codemix_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_clean_sinhala_df = test_clean_sinhala_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_misspell_df = test_misspell_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()

test_compound_df = test_compound_df[
    [TEXT_COLUMN, LABEL_COLUMN]
].copy()


# Remove missing values.

for df_name in [
    "train_df",
    "valid_df",
    "test_df",
    "test_transliteration_df",
    "test_codemix_df",
    "test_clean_sinhala_df",
    "test_misspell_df",
    "test_compound_df",
]:
    df = globals()[df_name]

    df.dropna(
        subset=[TEXT_COLUMN, LABEL_COLUMN],
        inplace=True
    )

    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
    df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

In [ ]:
# ============================================================
# 09. Encode labels
# ============================================================

label2id = {
    "non-hate": 0,
    "hate": 1,
}

id2label = {
    0: "non-hate",
    1: "hate",
}

train_df["labels"] = train_df[LABEL_COLUMN]
valid_df["labels"] = valid_df[LABEL_COLUMN]
test_df["labels"] = test_df[LABEL_COLUMN]

test_transliteration_df["labels"] = (
    test_transliteration_df[LABEL_COLUMN]
)

test_codemix_df["labels"] = (
    test_codemix_df[LABEL_COLUMN]
)

test_clean_sinhala_df["labels"] = (
    test_clean_sinhala_df[LABEL_COLUMN]
)

test_misspell_df["labels"] = (
    test_misspell_df[LABEL_COLUMN]
)

test_compound_df["labels"] = (
    test_compound_df[LABEL_COLUMN]
)

print("label2id:", label2id)
print("id2label:", id2label)

In [ ]:
# ============================================================
# 10. Convert DataFrames to Hugging Face datasets
# ============================================================

train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, "labels"]],
    preserve_index=False
)

test_transliteration_dataset = Dataset.from_pandas(
    test_transliteration_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

test_codemix_dataset = Dataset.from_pandas(
    test_codemix_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

test_clean_sinhala_dataset = Dataset.from_pandas(
    test_clean_sinhala_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

test_misspell_dataset = Dataset.from_pandas(
    test_misspell_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)

test_compound_dataset = Dataset.from_pandas(
    test_compound_df[
        [TEXT_COLUMN, "labels"]
    ],
    preserve_index=False
)


dataset = DatasetDict({
    "train": train_dataset,
    "validation": valid_dataset,
    "test": test_dataset,
})


other_test_dataset = DatasetDict({
    "transliteration_test":
        test_transliteration_dataset,

    "codemix_test":
        test_codemix_dataset,

    "clean_sinhala_test":
        test_clean_sinhala_dataset,

    "misspell_test":
        test_misspell_dataset,

    "compound_test":
        test_compound_dataset,
})


print(dataset)
print(other_test_dataset)

In [ ]:
# ============================================================
# 11. Load LaBSE
# ============================================================

labse = SentenceTransformer(
    MODEL_NAME
)

# Explicitly set maximum sequence length
# to match the other experiments.
labse.max_seq_length = MAX_LENGTH

print(labse)

print(
    "LaBSE maximum sequence length:",
    labse.max_seq_length
)

In [ ]:
# ============================================================
# 12. Load LaBSE tokenizer
# ============================================================

tokenizer = labse.tokenizer

print("Tokenizer loaded.")
print(
    "Vocabulary size:",
    tokenizer.vocab_size
)

In [ ]:
# ============================================================
# 13. Inspect tokenization
# ============================================================

sample_texts = train_df[
    TEXT_COLUMN
].head(10).tolist()

for text in sample_texts:

    tokens = tokenizer.tokenize(text)

    print("=" * 80)

    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(tokens)

    print("\nNUMBER OF TOKENS:")
    print(len(tokens))

In [ ]:
# ============================================================
# 14. Tokenize classification dataset
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
)

tokenized_test_dataset = other_test_dataset.map(
    tokenize_function,
    batched=True,
)

print(tokenized_dataset)
print(tokenized_test_dataset)

In [ ]:
# ============================================================
# 15. Data collator
# ============================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator created.")

In [ ]:
# ============================================================
# 16. LaBSE classification model
# ============================================================

class LaBSEForSequenceClassification(
    nn.Module
):

    def __init__(
        self,
        model_name,
        num_labels,
        max_length=128,
    ):

        super().__init__()

        # Load the complete LaBSE Sentence-Transformer.
        self.labse = SentenceTransformer(
            model_name
        )

        # Match experimental sequence length.
        self.labse.max_seq_length = max_length

        # LaBSE produces 768-dimensional sentence embeddings.
        self.embedding_dimension = 768

        self.num_labels = num_labels

        # Newly initialized classification head.
        self.classifier = nn.Linear(
            self.embedding_dimension,
            num_labels
        )


    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        labels=None,
        **kwargs,
    ):

        # Construct the feature dictionary expected
        # by SentenceTransformer.
        features = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        # LaBSE/BERT can accept token_type_ids,
        # although it is not always supplied.
        if token_type_ids is not None:

            features["token_type_ids"] = (
                token_type_ids
            )

        # Generate the LaBSE sentence embedding.
        outputs = self.labse(
            features
        )

        sentence_embedding = outputs[
            "sentence_embedding"
        ]

        # Classification head.
        logits = self.classifier(
            sentence_embedding
        )

        loss = None

        if labels is not None:

            loss_function = nn.CrossEntropyLoss()

            loss = loss_function(
                logits,
                labels
            )

        return {
            "loss": loss,
            "logits": logits,
        }


model = LaBSEForSequenceClassification(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    max_length=MAX_LENGTH,
)

print("LaBSE classification model created.")

In [ ]:
# ============================================================
# 17. Enable full fine-tuning
# ============================================================

for param in model.parameters():

    param.requires_grad = True


total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total parameters:     {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

In [ ]:
# ============================================================
# 18. Verify trainable parameters
# ============================================================

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():

    total_params += param.numel()

    if param.requires_grad:

        trainable_params += param.numel()

        print(
            "TRAINABLE:",
            name
        )


print("\n" + "=" * 60)

print(
    f"Total parameters:     {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)

print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

In [ ]:
# ============================================================
# 19. Evaluation metrics
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    # Accuracy
    accuracy = accuracy_score(
        labels,
        predictions
    )

    # Macro precision, recall, F1
    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    # Weighted F1
    _, _, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

In [ ]:
# ============================================================
# 20. Training configuration
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=5,

    # LaBSE is large, so start with a smaller batch.
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    # Effective batch size can be increased
    # without requiring more GPU memory.
    gradient_accumulation_steps=2,

    # Full fine-tuning learning rate.
    learning_rate=2e-5,

    # Regularization.
    weight_decay=0.01,

    # Evaluation.
    eval_strategy="epoch",

    # Save checkpoint after each epoch.
    save_strategy="epoch",

    # Keep best validation model.
    load_best_model_at_end=True,

    # Select best model using Macro F1.
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Logging.
    logging_strategy="steps",
    logging_steps=50,

    # Keep only the latest checkpoints.
    save_total_limit=2,

    # Mixed precision.
    fp16=torch.cuda.is_available(),

    # Reproducibility.
    seed=SEED,

    report_to="none",
)

In [ ]:
# ============================================================
# 21. Create Trainer
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("Trainer created.")

In [ ]:
# ============================================================
# 22. Train LaBSE
# ============================================================

train_result = trainer.train()

print("\nTraining completed.")

In [ ]:
# ============================================================
# 23. Validation evaluation
# ============================================================

validation_results = trainer.evaluate(
    tokenized_dataset["validation"]
)

print("\nValidation results:")

for key, value in validation_results.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 24. Final test evaluation
# ============================================================

test_results = trainer.evaluate(
    tokenized_dataset["test"]
)

print("\nTest results:")

for key, value in test_results.items():

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 25. Transliteration test
# ============================================================

test_results_on_transliteration = trainer.evaluate(
    tokenized_test_dataset[
        "transliteration_test"
    ]
)

print("\nTest on Transliteration:")

for key, value in (
    test_results_on_transliteration.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 26. Code-mix test
# ============================================================

test_results_on_codemix = trainer.evaluate(
    tokenized_test_dataset[
        "codemix_test"
    ]
)

print("\nTest on Code-Mix:")

for key, value in (
    test_results_on_codemix.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 27. Clean Sinhala test
# ============================================================

test_results_on_clean = trainer.evaluate(
    tokenized_test_dataset[
        "clean_sinhala_test"
    ]
)

print("\nTest on Clean Sinhala:")

for key, value in (
    test_results_on_clean.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 28. Misspelling test
# ============================================================

test_results_on_misspell = trainer.evaluate(
    tokenized_test_dataset[
        "misspell_test"
    ]
)

print("\nTest on Misspelling:")

for key, value in (
    test_results_on_misspell.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 29. Compound test
# ============================================================

test_results_on_compound = trainer.evaluate(
    tokenized_test_dataset[
        "compound_test"
    ]
)

print("\nTest on Compound:")

for key, value in (
    test_results_on_compound.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================================
# 30. Classification report - Main Test
# ============================================================

predictions = trainer.predict(
    tokenized_dataset["test"]
)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 31. Classification report - Transliteration
# ============================================================

predictions_tr = trainer.predict(
    tokenized_test_dataset[
        "transliteration_test"
    ]
)

predicted_labels_tr = np.argmax(
    predictions_tr.predictions,
    axis=-1
)

true_labels_tr = predictions_tr.label_ids

print(
    classification_report(
        true_labels_tr,
        predicted_labels_tr,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 32. Classification report - Code-Mix
# ============================================================

predictions_cm = trainer.predict(
    tokenized_test_dataset[
        "codemix_test"
    ]
)

predicted_labels_cm = np.argmax(
    predictions_cm.predictions,
    axis=-1
)

true_labels_cm = predictions_cm.label_ids

print(
    classification_report(
        true_labels_cm,
        predicted_labels_cm,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 33. Classification report - Clean Sinhala
# ============================================================

predictions_cs = trainer.predict(
    tokenized_test_dataset[
        "clean_sinhala_test"
    ]
)

predicted_labels_cs = np.argmax(
    predictions_cs.predictions,
    axis=-1
)

true_labels_cs = predictions_cs.label_ids

print(
    classification_report(
        true_labels_cs,
        predicted_labels_cs,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 34. Classification report - Misspelling
# ============================================================

predictions_misspell = trainer.predict(
    tokenized_test_dataset[
        "misspell_test"
    ]
)

predicted_labels_misspell = np.argmax(
    predictions_misspell.predictions,
    axis=-1
)

true_labels_misspell = (
    predictions_misspell.label_ids
)

print(
    classification_report(
        true_labels_misspell,
        predicted_labels_misspell,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 35. Classification report - Compound
# ============================================================

predictions_compound = trainer.predict(
    tokenized_test_dataset[
        "compound_test"
    ]
)

predicted_labels_compound = np.argmax(
    predictions_compound.predictions,
    axis=-1
)

true_labels_compound = (
    predictions_compound.label_ids
)

print(
    classification_report(
        true_labels_compound,
        predicted_labels_compound,
        target_names=[
            str(id2label[i])
            for i in range(NUM_LABELS)
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# 36. Confusion matrix
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

print("Confusion Matrix:")
print(cm)

In [ ]:
# ============================================================
# 37. Save fine-tuned LaBSE
# ============================================================

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

# Save the fine-tuned Sentence-Transformer.
trainer.model.labse.save(
    FINAL_MODEL_DIR
)

# Save classification head.
torch.save(
    trainer.model.classifier.state_dict(),
    os.path.join(
        FINAL_MODEL_DIR,
        "classifier_head.pt"
    )
)

# Save label configuration.
import json

label_config = {
    "label2id": label2id,
    "id2label": id2label,
    "num_labels": NUM_LABELS,
    "max_length": MAX_LENGTH,
}

with open(
    os.path.join(
        FINAL_MODEL_DIR,
        "label_config.json"
    ),
    "w"
) as f:

    json.dump(
        label_config,
        f,
        indent=4
    )

print("Model saved to:")
print(FINAL_MODEL_DIR)

In [ ]:
# ============================================================
# 38. Save experiment results
# ============================================================

experiment_results = {

    "experiment":
        "LaBSE Full Fine Tuning",

    "model":
        MODEL_NAME,

    "tokenizer":
        "LaBSE tokenizer",

    "encoder_trainable":
        True,

    "classification_head_trainable":
        True,

    "max_length":
        MAX_LENGTH,

    "seed":
        SEED,

    "total_parameters":
        total_params,

    "trainable_parameters":
        trainable_params,

    "trainable_percentage":
        (
            100 *
            trainable_params /
            total_params
        ),

    # Main test
    "test_accuracy":
        test_results.get(
            "eval_accuracy"
        ),

    "test_macro_precision":
        test_results.get(
            "eval_macro_precision"
        ),

    "test_macro_recall":
        test_results.get(
            "eval_macro_recall"
        ),

    "test_macro_f1":
        test_results.get(
            "eval_macro_f1"
        ),

    "test_weighted_f1":
        test_results.get(
            "eval_weighted_f1"
        ),

    # Robustness tests
    "transliteration_macro_f1":
        test_results_on_transliteration.get(
            "eval_macro_f1"
        ),

    "codemix_macro_f1":
        test_results_on_codemix.get(
            "eval_macro_f1"
        ),

    "clean_sinhala_macro_f1":
        test_results_on_clean.get(
            "eval_macro_f1"
        ),

    "misspell_macro_f1":
        test_results_on_misspell.get(
            "eval_macro_f1"
        ),

    "compound_macro_f1":
        test_results_on_compound.get(
            "eval_macro_f1"
        ),
}


results_df = pd.DataFrame(
    [experiment_results]
)

display(results_df)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Results saved.")

In [ ]:
# ============================================================
# 39. Test individual examples
# ============================================================

def predict_text(text):

    model.eval()

    # Tokenize.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move inputs to model device.
    device = next(
        model.parameters()
    ).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model(
            **inputs
        )

    probabilities = torch.softmax(
        outputs["logits"],
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    return {
        "text": text,

        "prediction":
            id2label[predicted_id],

        "confidence":
            probabilities[
                0,
                predicted_id
            ].item(),
    }


# Test several examples.

for text in test_df[
    TEXT_COLUMN
].head(10):

    result = predict_text(text)

    print("=" * 80)

    print(result)

In [ ]:
# ============================================================
# 40. Final experiment summary
# ============================================================

print("=" * 70)
print("LaBSE FULL FINE-TUNING SUMMARY")
print("=" * 70)

print("\nModel:")
print(MODEL_NAME)

print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))

print("\nMain Test:")
print(
    "Accuracy:",
    test_results["eval_accuracy"]
)

print(
    "Macro Precision:",
    test_results["eval_macro_precision"]
)

print(
    "Macro Recall:",
    test_results["eval_macro_recall"]
)

print(
    "Macro F1:",
    test_results["eval_macro_f1"]
)

print(
    "Weighted F1:",
    test_results["eval_weighted_f1"]
)

print("\nRobustness Macro F1:")

print(
    "Transliteration:",
    test_results_on_transliteration[
        "eval_macro_f1"
    ]
)

print(
    "Code-Mix:",
    test_results_on_codemix[
        "eval_macro_f1"
    ]
)

print(
    "Clean Sinhala:",
    test_results_on_clean[
        "eval_macro_f1"
    ]
)

print(
    "Misspelling:",
    test_results_on_misspell[
        "eval_macro_f1"
    ]
)

print(
    "Compound:",
    test_results_on_compound[
        "eval_macro_f1"
    ]
)